# 3. Anonymization

Six rules, in order: outlet signature in the titles → `[OUTLET]`; e-mails → `[EMAIL]`; links → `[URL]`;
@accounts → `[ACCOUNT]`; the outlet's own authors → `[AUTHOR]`; the outlet name from its domain → `[OUTLET]`
(see `anonymization.py`). The checks are automatic.

In [1]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import re

import anonymization as AN

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


In [2]:
art = pd.read_csv(C.OUT / "articles.csv.gz")
anon, counts, found = AN.anonymize(art)
RULES = ["signature", "email", "url", "account", "author", "name"]
print(f"articles with at least one replacement: {((counts[RULES].sum(axis=1) + counts[[f'title_{r}' for r in RULES]].sum(axis=1)) > 0).mean():.1%}")
pd.DataFrame({"articles touched (%)": [(counts[r] > 0).mean() * 100 for r in RULES],
              "replacements": [counts[r].sum() for r in RULES],
              "titles touched (%)": [(counts[f"title_{r}"] > 0).mean() * 100 for r in RULES]}, index=RULES).round(1)

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


articles with at least one replacement: 54.6%


,articles touched (%),replacements,titles touched (%)
signature,6.5,4450,21.6
email,1.3,385,0.0
url,14.5,7860,0.8
account,9.9,7807,0.0
author,11.3,5235,1.3
name,17.4,8727,1.0


In [3]:
print("common-word names:", ", ".join(sorted(found["common"])))
print("written in lowercase by their own outlet:", ", ".join(sorted(found["lowercase"])))
print(f"signatures: {len(found['signatures'])} outlets; own authors: {sum(map(len, found['authors'].values())):,}")
for token in ["[OUTLET]", "[AUTHOR]"]:
    for e in AN.examples(anon.text, re.compile(re.escape(token)), n=3):
        print(" ·", e)

common-word names: antiwar, bulwark, bureau, dispatch, fee, governing, guardian, mic, nation, national, observer, politics, popular, progressive, racket, reason, spiked, time, week
written in lowercase by their own outlet: spiked
signatures: 54 outlets; own authors: 4,597
 · e Czech-based developer of the game, said in a statement to [OUTLET]. While misinformation is accelerating faster than ever than
 ·  examine statements from the authorities and media reports. [OUTLET] confirmed several incidents in which villages were engulfed
 · y of Technology Keep Independent Journalism Alive – Support [OUTLET] Dear Reader, Since 2013, The Australian Independent Media N


 · the past three years. He writes here in a personal capacity [AUTHOR] is the chief product officer at UK wealth-tech company Time
 · rty when actually in power Subscribers Get More from JOSIAH [AUTHOR] also writes the On the Ground column, exclusive to the prin
 · t. TPM will be covering developments all day today: - TPM’s [AUTHOR]: Federal Judge Denies ‘Extraordinary Measure’ of Enjoining 


## Check: nothing left

In [4]:
left = AN.leftovers(anon, found)
distinct = left[~left.common_word]
print(f"distinctive names ({len(distinct)} outlets): articles with the name as a word {distinct.left_as_word.sum()}, "
      f"inside other words {distinct.left_inside_words.sum()}")
assert distinct.left_as_word.sum() == 0
left[left.common_word | (left.left_inside_words > 0)]

distinctive names (208 outlets): articles with the name as a word 0, inside other words 3


,domain,names,signature,common_word,articles,left_as_word,left_inside_words
15,antiwar.com,antiwar,,True,18,0,0
28,cf.org,cf,,False,198,0,2
59,fee.org,fee,,True,162,4,43
72,governing.com,governing,,True,158,10,10
109,mic.com,mic,,True,59,0,26
130,observer.com,observer,,True,108,0,0
139,politics.co.uk,politics,Politics.co.uk,True,172,56,57
143,popular.info,popular,,True,190,14,26
148,progressive.org,progressive,,True,20,4,5
151,quillette.com,quillette,,False,134,0,1


In [5]:
anon[["article_id", "domain", "title", "date", "text", "score", "label", "level"]].to_csv(C.OUT / "articles_anonymous.csv.gz", index=False)
left.to_csv(C.OUT / "anonymization_check.csv", index=False)
print("saved", C.OUT / "articles_anonymous.csv.gz")

saved /home/fabio.pinelli/John/Dataset/tentativi/20_FromScratch/articles_anonymous.csv.gz
